# PERFORM media transcriber (Google Colab)\nPut media in `MyDrive/PERFORM Learning/input`. This notebook writes resumable transcript files to `MyDrive/PERFORM Learning/transcripts`. Run it in a Colab runtime with a GPU when one is available.

In [ ]:
!pip -q install faster-whisper\nfrom google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
from pathlib import Path\nimport json\nfrom faster_whisper import WhisperModel\n\nINPUT = Path('/content/drive/MyDrive/PERFORM Learning/input')\nOUTPUT = Path('/content/drive/MyDrive/PERFORM Learning/transcripts')\nMODEL = 'small'  # tiny, base, small, medium; increase only when accuracy needs it\nLANGUAGE = None  # e.g. 'en'; None detects automatically\nINPUT.mkdir(parents=True, exist_ok=True)\nOUTPUT.mkdir(parents=True, exist_ok=True)\nMEDIA_EXTENSIONS = {'.mp3', '.m4a', '.wav', '.aac', '.flac', '.ogg', '.mp4', '.mov', '.mkv', '.webm'}\nprint(f'Put files in: {INPUT}')\nprint(f'Transcripts go to: {OUTPUT}')

In [ ]:
def timestamp(seconds, srt=False):\n    milliseconds = round(seconds * 1000)\n    hours, milliseconds = divmod(milliseconds, 3_600_000)\n    minutes, milliseconds = divmod(milliseconds, 60_000)\n    seconds, milliseconds = divmod(milliseconds, 1000)\n    separator = ',' if srt else '.'\n    return f'{hours:02d}:{minutes:02d}:{seconds:02d}{separator}{milliseconds:03d}'\n\ndef write_outputs(media, segments, info):\n    stem = OUTPUT / media.stem\n    text = '\n'.join(segment.text.strip() for segment in segments)\n    stem.with_suffix('.txt').write_text(text + '\n', encoding='utf-8')\n    with stem.with_suffix('.srt').open('w', encoding='utf-8') as file:\n        for index, segment in enumerate(segments, 1):\n            file.write(f'{index}\n{timestamp(segment.start, True)} --> {timestamp(segment.end, True)}\n{segment.text.strip()}\n\n')\n    with stem.with_suffix('.vtt').open('w', encoding='utf-8') as file:\n        file.write('WEBVTT\n\n')\n        for segment in segments:\n            file.write(f'{timestamp(segment.start)} --> {timestamp(segment.end)}\n{segment.text.strip()}\n\n')\n    payload = {'source': media.name, 'language': info.language, 'language_probability': info.language_probability, 'segments': [segment._asdict() for segment in segments]}\n    stem.with_suffix('.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')\n\ndevice = 'cuda'  # Change to 'cpu' only if Colab gives no GPU\ncompute_type = 'float16' if device == 'cuda' else 'int8'\nmodel = WhisperModel(MODEL, device=device, compute_type=compute_type)\nmedia_files = [path for path in INPUT.rglob('*') if path.suffix.lower() in MEDIA_EXTENSIONS]\nfor media in media_files:\n    completed = OUTPUT / f'{media.stem}.json'\n    if completed.exists():\n        print(f'Skipping completed: {media.name}')\n        continue\n    print(f'Transcribing: {media.name}')\n    segment_iterator, info = model.transcribe(str(media), language=LANGUAGE, vad_filter=True)\n    segments = list(segment_iterator)\n    write_outputs(media, segments, info)\n    print(f'Done: {media.name} ({info.language})')